In [5]:
import os
import bs4
from dotenv import load_dotenv
from uuid import uuid4

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [6]:
tracer = LangChainTracer()

# Multi-representation Indexing

In [7]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [8]:
print(len(docs))
print(docs[0].page_content[500:1000])


2
Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
A


In [9]:
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatGroq(model="openai/gpt-oss-20b", temperature=0)
    | StrOutputParser()
)

for doc in docs:
    doc.page_content = doc.page_content[:4000]

summaries = [chain.invoke(doc, config={"callbacks": [tracer]}) for doc in docs]

print(summaries)

['**LLM‑Powered Autonomous Agents – Summary**\n\nLilian\u202fWeng’s June\u202f2023 post outlines how large language models (LLMs) can serve as the “brain” of autonomous agents, complementing three core modules: **planning**, **memory**, and **tool use**.\n\n1. **Planning** – The agent breaks a complex task into sub‑goals and refines its actions through self‑reflection. Techniques include:\n   * **Chain‑of‑Thought (CoT)** prompting, which forces the model to reason step‑by‑step.\n   * **Tree‑of‑Thoughts** (Yao\u202fet\u202fal.,\u202f2023), which explores multiple reasoning paths in a tree structure, evaluated by a classifier or majority vote.\n   * **LLM+P** (Liu\u202fet\u202fal.,\u202f2023), where the LLM translates a problem into PDDL, a classical planner generates a plan, and the plan is translated back into natural language.\n\n2. **Memory** – Two layers:\n   * **Short‑term memory** via in‑context learning (prompt engineering).\n   * **Long‑term memory** using an external vector sto

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

id_key = "doc_id"
doc_ids = [str(uuid4()) for _ in docs]

summary_docs = [
    Document(page_content=summary, metadata={id_key: doc_id})
    for summary, doc_id in zip(summaries, doc_ids)
]

vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=embeddings,
    persist_directory="chroma",
)
vectorstore.add_documents(summary_docs, ids=doc_ids)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['bd04c24b-a453-4799-ba40-5f998394dab2',
 'f7001842-a7f4-42a0-9fca-9cf76b781eac']

In [11]:
from langchain_classic.retrievers import MultiVectorRetriever

store = InMemoryStore()
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
    search_kwargs={"k": 4},
)

store.mset(list(zip(doc_ids, docs)))

retrieved_docs = retriever.invoke("How do agents use tools?")
print(len(retrieved_docs))
print(retrieved_docs[0].page_content[:500])

2






LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: 


# RAPTOR